# Text to Video generation with Wan2.1 and OpenVINO

 Wan2.1 is a comprehensive and open suite of video foundation models that pushes the boundaries of video generation.

 Built upon the mainstream diffusion transformer paradigm, Wan 2.1 achieves significant advancements in generative capabilities through a series of innovations, including our novel spatio-temporal variational autoencoder (VAE), scalable pre-training strategies, large-scale data construction, and automated evaluation metrics. These contributions collectively enhance the model's performance and versatility.

 You can find more details about model in [model card](https://huggingface.co/Wan-AI/Wan2.1-T2V-1.3B) and [original repository](https://github.com/Wan-Video/Wan2.1)

 In this tutorial we consider how to convert, optimize and run Wan2.1 model using OpenVINO.
 Additionally, for achieving inference speedup, we will apply [CausVid](https://causvid.github.io/) distillation approach using LoRA.

 ![](https://causvid.github.io/images/methods.jpg)

 Current video diffusion models achieve impressive generation quality but struggle in interactive applications due to bidirectional attention dependencies. The generation of a single frame requires the model to process the entire sequence, including the future. CausVid address this limitation by adapting a pretrained bidirectional diffusion transformer to an autoregressive transformer that generates frames on-the-fly. To further reduce latency, the authors extend distribution matching distillation (DMD) to videos, distilling 50-step diffusion model into a 4-step generator.

 The method distills a many-step, bidirectional video diffusion model into a 4-step, causal generator. The training process consists of two stages: 
 1. Student Initialization: Initialization of the causal student by pretraining it on a small set of ODE solution pairs generated by the bidirectional teacher. This step helps stabilize the subsequent distillation training.
 2. Asymmetric Distillation: Using the bidirectional teacher model, we train the causal student generator through a distribution matching distillation loss. 

More details about CausVid can be found in the [paper](https://arxiv.org/abs/2412.07772), [original repository](https://github.com/tianweiy/CausVid) and [project page](https://causvid.github.io/)
#### Table of contents:

- [Prerequisites](#Prerequisites)
- [Convert model to OpenVINO Intermediate Representation](#Convert-model-to-OpenVINO-Intermediate-Representation)
    - [Compress model weights](#Compress-model-weights)
- [Prepare model inference pipeline](#Prepare-model-inference-pipeline)
    - [Select inference device](#Select-inference-device)
- [Run OpenVINO Model Inference](#Run-OpenVINO-Model-Inference)
- [Interactive demo](#Interactive-demo)


### Installation Instructions

This is a self-contained example that relies solely on its own code.

We recommend  running the notebook in a virtual environment. You only need a Jupyter server to start.
For details, please refer to [Installation Guide](https://github.com/openvinotoolkit/openvino_notebooks/blob/latest/README.md#-installation-guide).

<img referrerpolicy="no-referrer-when-downgrade" src="https://static.scarf.sh/a.png?x-pxid=5b5a4db0-7875-4bfb-bdbd-01698b5b1a77&file=notebooks/wan2.1-text-to-video/wan2.1-text-to-video.ipynb" />


## Prerequisites
[back to top ⬆️](#Table-of-contents:)

In [1]:
import platform

%pip install -q "torch>=2.1" "git+https://github.com/huggingface/diffusers.git" "transformers>=4.49.0" "accelerate" "safetensors" "sentencepiece" "peft>=0.7.0" "ftfy" "gradio>=4.19" "opencv-python" --extra-index-url https://download.pytorch.org/whl/cpu
%pip install --pre -U "openvino>=2025.1.0" "nncf>=2.16.0" --extra-index-url https://storage.openvinotoolkit.org/simple/wheels/nightly

if platform.system() == "Darwin":
    %pip install -q "numpy<2.0.0"

In [2]:
import requests
from pathlib import Path

if not Path("ov_wan_helper.py").exists():
    r = requests.get(url="https://raw.githubusercontent.com/openvinotoolkit/openvino_notebooks/latest/notebooks/wan2.1-text-to-video/ov_wan_helper.py")
    open("ov_wan_helper.py", "w").write(r.text)

if not Path("gradio_helper.py").exists():
    r = requests.get(url="https://raw.githubusercontent.com/openvinotoolkit/openvino_notebooks/latest/notebooks/wan2.1-text-to-video/gradio_helper.py")
    open("gradio_helper.py", "w").write(r.text)

if not Path("notebook_utils.py").exists():
    r = requests.get(url="https://raw.githubusercontent.com/openvinotoolkit/openvino_notebooks/latest/utils/notebook_utils.py")
    open("notebook_utils.py", "w").write(r.text)

## Convert model to OpenVINO Intermediate Representation
[back to top ⬆️](#Table-of-contents:)


Wan2.1 is PyTorch model. OpenVINO supports PyTorch models via conversion to OpenVINO Intermediate Representation (IR). [OpenVINO model conversion API](https://docs.openvino.ai/2024/openvino-workflow/model-preparation.html#convert-a-model-with-python-convert-model) should be used for these purposes. `ov.convert_model` function accepts original PyTorch model instance and example input for tracing and returns `ov.Model` representing this model in OpenVINO framework. Converted model can be used for saving on disk using `ov.save_model` function or directly loading on device using `core.complie_model`.

Model consist of 3 parts:
* **Text Encoder** to encode input multi-language text, incorporating cross-attention within each transformer block to embed the text into the model structure
* **Diffusion Transformer** for step by step denoising of generated video guided by text instructions.
* **VAE Decoder** to decode generated video represented in latent space.

Model performs text-to-video generation task. For preserving original model flexibility, we will convert each part separately.

The script `ov_wan_helper.py` contains helper function for model conversion, please check its content if you interested in conversion details.

### Compress model weights
[back to top ⬆️](#Table-of-contents:)

For reducing memory consumption, weights compression optimization can be applied using [NNCF](https://github.com/openvinotoolkit/nncf). 

<details>
    <summary><b>Click here for more details about weight compression</b></summary>
Weight compression aims to reduce the memory footprint of a model. It can also lead to significant performance improvement for large memory-bound models, such as Large Language Models (LLMs). LLMs and other models, which require extensive memory to store the weights during inference, can benefit from weight compression in the following ways:

* enabling the inference of exceptionally large models that cannot be accommodated in the memory of the device;

* improving the inference performance of the models by reducing the latency of the memory access when computing the operations with weights, for example, Linear layers.

[Neural Network Compression Framework (NNCF)](https://github.com/openvinotoolkit/nncf) provides 4-bit / 8-bit mixed weight quantization as a compression method primarily designed to optimize LLMs. The main difference between weights compression and full model quantization (post-training quantization) is that activations remain floating-point in the case of weights compression which leads to a better accuracy. Weight compression for LLMs provides a solid inference performance improvement which is on par with the performance of the full model quantization. In addition, weight compression is data-free and does not require a calibration dataset, making it easy to use.

`nncf.compress_weights` function can be used for performing weights compression. The function accepts an OpenVINO model and other compression parameters. Compared to INT8 compression, INT4 compression improves performance even more, but introduces a minor drop in prediction quality.

More details about weights compression, can be found in [OpenVINO documentation](https://docs.openvino.ai/2024/openvino-workflow/model-optimization-guide/weight-compression.html).
</details>

In [3]:
import ipywidgets as widgets

# Read more about telemetry collection at https://github.com/openvinotoolkit/openvino_notebooks?tab=readme-ov-file#-telemetry
from notebook_utils import collect_telemetry

collect_telemetry("wan2.1-text-to-video.ipynb")

model_id = "Wan-AI/Wan2.1-T2V-1.3B-Diffusers"
model_base_dir = Path(model_id.split("/")[-1])

model_format = widgets.Dropdown(
    options=["FP16", "INT8", "INT4"],
    value="INT4",
    description="Model format:",
)

model_format

Dropdown(description='Model format:', index=2, options=('FP16', 'INT8', 'INT4'), value='INT4')

In [4]:
import nncf

model_dir = model_base_dir / model_format.value

if model_format.value == "INT4":
    weights_compression_config = {"mode": nncf.CompressWeightsMode.INT4_ASYM, "group_size": 64, "ratio": 1.0}
elif model_format.value == "INT8":
    weights_compression_config = {"mode": nncf.CompressWeightsMode.INT8_ASYM}
else:
    weights_compression_config = None

In [5]:
from ov_wan_helper import convert_pipeline

# Uncomment the line to see model conversion code
# ??convert_pipeline

Multiple distributions found for package optimum. Picked distribution: optimum
The installed version of bitsandbytes was compiled without GPU support. 8-bit optimizers, 8-bit multiplication, and GPU quantization are unavailable.


In [6]:
convert_pipeline(model_id, model_dir, compression_config=weights_compression_config)

✅ Wan-AI/Wan2.1-T2V-1.3B-Diffusers model already converted. You can find results in Wan2.1-T2V-1.3B-Diffusers/INT4


## Prepare model inference pipeline
[back to top ⬆️](#Table-of-contents:)


`OVWanPipeline` defined in `ov_wan_helper.py` provides unified interface for running model inference. It accepts model directory and target device map for inference.

In [7]:
from ov_wan_helper import OVWanPipeline

# Uncomment the line to see model inference code
# ??OVWanPipeline

### Select inference device
[back to top ⬆️](#Table-of-contents:)

You can specify inference device for each pipeline component or use the same device for all of them using widgets below.

In [8]:
from notebook_utils import device_widget

device_transformer = device_widget(exclude=["NPU"], description="Transformer")
device_text_encoder = device_widget(exclude=["NPU"], description="Text Encoder")
device_vae = device_widget(exclude=["NPU"], description="VAE Decoder")

devices = widgets.VBox([device_transformer, device_text_encoder, device_vae])

devices

In [9]:
device_map = {"transformer": device_transformer.value, "text_encoder": device_text_encoder.value, "vae": device_vae.value}

ov_pipe = OVWanPipeline(model_dir, device_map)

## Run OpenVINO Model Inference
[back to top ⬆️](#Table-of-contents:)

In [ ]:
from diffusers.utils import export_to_video

prompt = "A cat walks on the grass, realistic"
negative_prompt = "Bright tones, overexposed, static, blurred details, subtitles, style, works, paintings, images, static, overall gray, worst quality, low quality, JPEG compression residue, ugly, incomplete, extra fingers, poorly drawn hands, poorly drawn faces, deformed, disfigured, misshapen limbs, fused fingers, still picture, messy background, three legs, many people in the background, walking backwards"

output = ov_pipe(prompt=prompt, negative_prompt=negative_prompt, height=480, width=832, num_frames=20, guidance_scale=1.0, num_inference_steps=4).frames[0]
export_to_video(output, "output.mp4", fps=10)

In [11]:
from IPython.display import Video

display(Video("output.mp4"))

## Interactive demo
[back to top ⬆️](#Table-of-contents:)

In [ ]:
from gradio_helper import make_demo

demo = make_demo(ov_pipe)

try:
    demo.launch(debug=True)
except Exception:
    demo.launch(share=True, debug=True)
# if you are launching remotely, specify server_name and server_port
# demo.launch(server_name='your server name', server_port='server port in int')
# Read more in the docs: https://gradio.app/docs/